In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [44]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# Extract text from PDF files
def load_pdf_files(data):
    loader = DirectoryLoader(
        new_data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [ ]:
extracted_data = load_pdf_files("../data/new_data")

In [ ]:
extracted_data

In [48]:

loader = PyPDFLoader("../data/Bookshelf_NBK22183.pdf")

genes_documents = loader.load()

In [13]:
def text_split(docs):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
    text_chunk=text_splitter.split_documents(docs)
    return text_chunk

In [49]:
texts_chunk = text_split(genes_documents)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 521


In [51]:
print(texts_chunk[0])

page_content='Genes and Disease
Last Updated: January 31, 2011
National Center for Biotechnology Information (US)
Bethesda (MD)
National Center for Biotechnology Information (US)' metadata={'producer': 'Antenna House PDF Output Library 7.2.1732', 'creator': 'AH Formatter V7.2 MR3 for Linux64 : 7.2.4.55390 (2022-01-31T09:48+09)', 'creationdate': '2022-11-14T02:19:14-05:00', 'moddate': '2022-11-14T02:19:14-05:00', 'title': 'Genes and Disease', 'trapped': '/False', 'source': '../data/Bookshelf_NBK22183.pdf', 'total_pages': 221, 'page': 0, 'page_label': 'i'}


In [ ]:
for chunk in texts_chunk:
    chunk.metadata["source_name"] = "Genes and Disease"

docsearch.add_documents(
    documents=texts_chunk
)

In [19]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return embeddings

In [ ]:
embedding = download_embeddings()

In [ ]:
vector=embedding.embed_query("Hello World")
vector

In [25]:
from dotenv import load_dotenv
import os
load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [26]:
from pinecone import Pinecone 
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)


In [27]:
from pinecone import ServerlessSpec 

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )


index = pc.Index(index_name)

In [28]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [32]:
# Load Existing index 

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [53]:
index = pc.Index("medical-chatbot")

stats = index.describe_index_stats()

print(stats)

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 6483}},
 'total_vector_count': 6483,
 'vector_type': 'dense'}


In [30]:
from langchain_core.documents import Document
dswith = Document(
    page_content="dswithbappy is a youtube channel that provides tutorials on various topics.",
    metadata={"source": "Youtube"}
)

In [31]:
docsearch.add_documents(documents=[dswith])

['cf5de196-f032-4a85-97f5-62a80288eb7a']

In [33]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [ ]:
retrieved_docs=retriever.invoke("What is Acne?")
retrieved_docs

In [54]:
docs = retriever.invoke("What is epilepsy?")

for i, doc in enumerate(docs):
    print(f"\n========== Document {i + 1} ==========")
    print("Content:")
    print(doc.page_content[:1000])
    print("\nMetadata:")
    print(doc.metadata)


========== Document 1 ==========
Content:
Epilepsy
Epilepsy affects approximately 1% of the population making it one of the most common neurological diseases. 
Epilepsy can strike at any time of life—from infancy to old age. While epilepsy varies widely in type and severity, 
all forms of this disorder are characterized by recurring seizures resulting from abnormal cell firing in the brain. 
In approximately 30% of cases, epilepsy is caused by such events as head trauma, tumor, stroke, or infection. In

Metadata:
{'creationdate': '2022-11-14T02:19:14-05:00', 'creator': 'AH Formatter V7.2 MR3 for Linux64 : 7.2.4.55390 (2022-01-31T09:48+09)', 'moddate': '2022-11-14T02:19:14-05:00', 'page': 108.0, 'page_label': '103', 'producer': 'Antenna House PDF Output Library 7.2.1732', 'source': '../data/Bookshelf_NBK22183.pdf', 'source_name': 'Genes and Disease', 'title': 'Genes and Disease', 'total_pages': 221.0, 'trapped': '/False'}

========== Document 2 ==========
Content:
imagined stress, dang

In [36]:
from langchain_google_genai import ChatGoogleGenerativeAI

chatModel = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0
)

In [38]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [39]:
prompt = ChatPromptTemplate.from_template("""
You are a medical assistant.

Answer the question based only on the following context.

<context>
{context}
</context>

Question:
{input}

If the answer is not present in the context, say:
"I don't have enough information in the provided medical documents."
""")

In [40]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [41]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough()
    }
    | prompt
    | chatModel
    | StrOutputParser()
)

In [45]:
response = rag_chain.invoke(
    "how to cure acne?"
)

print(response)

Based on the provided context, alternative treatments and recommendations for acne include:

* **Proper cleansing:** Keep the skin oil-free and use noncomedogenic makeup and moisturizers.
* **Dietary changes:** Eat a well-balanced diet that is high in fiber, zinc, and raw foods, while avoiding foods that trigger flare-ups, alcohol, dairy products, smoking, caffeine, sugar, processed foods, and foods high in iodine (such as salt).
* **Supplementation:** Use herbs such as burdock root (*Arctium lappa*), red clover (*Trifolium pratense*), milk thistle (*Silybum marianum*), cnidium seed (*Cnidium monnieri*), and honeysuckle flower (*Lonicera japonica*), as well as nutrients like essential fatty acids, vitamin B complex, zinc, vitamin A, and chromium (proper amounts should be recommended by wholistic physicians or nutritionists).
* **Lifestyle and skincare habits:** Shampoo often, wear hair off the face, give dry pimples a limited amount of sun exposure (unless told otherwise), do not pick 